# 03 - Commute Period Delay Analysis

This notebook compares delay patterns across different commute periods.

**Features:**
- Side-by-side comparison of morning, evening, and off-peak
- Grouped bar charts
- Weekend vs weekday analysis

**Output:** `static/plots/commute_delay.html`

In [1]:
import os
import sys
import sqlite3
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta

project_root = os.path.abspath(os.path.join(os.getcwd(), '..','..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

DB_PATH = os.path.join(project_root, 'data', 'caltrain_lat_long.db')
GTFS_PATH = os.path.join(project_root, 'gtfs_data')
OUTPUT_DIR = os.path.join(project_root, 'static', 'plots')

## Configuration

In [2]:
# ====================
# CONFIGURATION
# ====================

DAYS_TO_ANALYZE = 30

# Commute period definitions (hours)
PERIODS = {
    'Morning Commute': (6, 9),
    'Midday': (9, 15),
    'Evening Commute': (15, 19),
    'Night': (19, 23),
}

STATUS_COLORS = {
    'On Time': '#00CC96',
    'Minor': '#FECB52',
    'Major': '#EF553B',
}

PERIOD_COLORS = {
    'Morning Commute': '#ff9f43',
    'Midday': '#54a0ff',
    'Evening Commute': '#5f27cd',
    'Night': '#222f3e',
}

THEME = {
    'bg_color': '#0d1117',
    'text_color': '#c9d1d9',
    'grid_color': '#21262d',
}

FIGURE_WIDTH = 1000
FIGURE_HEIGHT = 500

## Load Data

In [3]:
sys.path.insert(0, os.path.join(project_root, 'src'))
from utils.time_utils import calculate_time_difference, normalize_time
from utils.geo_utils import haversine

def categorize_period(hour):
    for period, (start, end) in PERIODS.items():
        if start <= hour < end:
            return period
    return 'Night'

def load_arrivals():
    conn = sqlite3.connect(DB_PATH)
    cutoff = (datetime.now() - timedelta(days=DAYS_TO_ANALYZE)).strftime('%Y-%m-%d')
    query = f"SELECT trip_id, stop_id, vehicle_lat, vehicle_lon, timestamp FROM train_locations WHERE timestamp >= '{cutoff}'"
    raw_df = pd.read_sql_query(query, conn)
    conn.close()
    
    if raw_df.empty:
        return pd.DataFrame()
    
    raw_df['timestamp'] = pd.to_datetime(raw_df['timestamp'], errors='coerce')
    raw_df = raw_df.dropna(subset=['timestamp'])
    
    stops_df = pd.read_csv(os.path.join(GTFS_PATH, 'stops.txt'))
    stops_df = stops_df[stops_df['stop_id'].str.isnumeric()]
    stop_times_df = pd.read_csv(os.path.join(GTFS_PATH, 'stop_times.txt'))
    
    raw_df['stop_id'] = pd.to_numeric(raw_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    raw_df['trip_id'] = pd.to_numeric(raw_df['trip_id'].astype(str), errors='coerce').astype('Int64')
    stops_df['stop_id'] = pd.to_numeric(stops_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    stop_times_df['stop_id'] = pd.to_numeric(stop_times_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    stop_times_df['trip_id'] = pd.to_numeric(stop_times_df['trip_id'].astype(str), errors='coerce').astype('Int64')
    raw_df = raw_df.dropna(subset=['trip_id', 'stop_id'])
    
    df = pd.merge(raw_df, stop_times_df[['trip_id', 'stop_id', 'arrival_time']], on=['trip_id', 'stop_id'], how='inner')
    df = pd.merge(df, stops_df[['stop_id', 'stop_lat', 'stop_lon']], on='stop_id', how='inner')
    
    df['distance'] = df.apply(lambda r: haversine(r['vehicle_lat'], r['vehicle_lon'], r['stop_lat'], r['stop_lon']), axis=1)
    df['date'] = df['timestamp'].dt.date
    df['hour'] = df['timestamp'].dt.hour
    df['arrival_time'] = df['arrival_time'].apply(normalize_time)
    
    min_dist = df.groupby(['trip_id', 'stop_id', 'date'])['distance'].min().reset_index()
    merged = pd.merge(df, min_dist, on=['trip_id', 'stop_id', 'date', 'distance'])
    arrivals = merged.groupby(['trip_id', 'stop_id', 'date']).first().reset_index()
    
    arrivals['delay_min'] = arrivals.apply(lambda r: calculate_time_difference(r['arrival_time'], r['timestamp']), axis=1)
    arrivals.loc[arrivals.delay_min > 500, 'delay_min'] = 0
    arrivals.loc[arrivals.delay_min < -100, 'delay_min'] = 0
    
    arrivals['status'] = 'On Time'
    arrivals.loc[(arrivals.delay_min > 4) & (arrivals.delay_min <= 15), 'status'] = 'Minor'
    arrivals.loc[arrivals.delay_min > 15, 'status'] = 'Major'
    
    arrivals['period'] = arrivals['hour'].apply(categorize_period)
    arrivals['is_weekend'] = pd.to_datetime(arrivals['date']).dt.dayofweek >= 5
    
    print(f"Processed {len(arrivals):,} arrivals")
    return arrivals

arrivals = load_arrivals()

Base directory: D:\caltrain-tracker
Dotenv path: D:\caltrain-tracker\.env
Dotenv file exists: True
Processed 48,518 arrivals


## Calculate Period Statistics

In [4]:
def calculate_period_stats(arrivals):
    stats = arrivals.groupby(['period', 'status']).size().reset_index(name='count')
    pivot = stats.pivot(index='period', columns='status', values='count').fillna(0)
    
    for s in ['On Time', 'Minor', 'Major']:
        if s not in pivot.columns:
            pivot[s] = 0
    
    pivot = pivot[['On Time', 'Minor', 'Major']]
    pivot['total'] = pivot.sum(axis=1)
    pivot['on_time_pct'] = (pivot['On Time'] / pivot['total'] * 100).round(1)
    pivot = pivot.reset_index()
    
    # Order by time of day
    period_order = list(PERIODS.keys())
    pivot['order'] = pivot['period'].apply(lambda x: period_order.index(x) if x in period_order else 99)
    pivot = pivot.sort_values('order').drop('order', axis=1)
    
    return pivot

period_stats = calculate_period_stats(arrivals)
period_stats

status,period,On Time,Minor,Major,total,on_time_pct
2,Morning Commute,7565,433,21,8019,94.3
1,Midday,14013,842,103,14958,93.7
0,Evening Commute,11277,1195,56,12528,90.0
3,Night,12014,744,255,13013,92.3


## Create Charts

In [5]:
def create_commute_chart(period_stats, arrivals):
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=['Arrivals by Period', 'On-Time % by Period'],
        specs=[[{"type": "bar"}, {"type": "bar"}]]
    )
    
    # Left: Stacked bar by status
    for status in ['On Time', 'Minor', 'Major']:
        fig.add_trace(
            go.Bar(
                x=period_stats['period'],
                y=period_stats[status],
                name=status,
                marker_color=STATUS_COLORS[status],
                hovertemplate=f'{status}: %{{y:,.0f}}<extra></extra>',
            ),
            row=1, col=1
        )
    
    # Right: On-time percentage bars
    colors = [PERIOD_COLORS.get(p, '#888') for p in period_stats['period']]
    fig.add_trace(
        go.Bar(
            x=period_stats['period'],
            y=period_stats['on_time_pct'],
            name='On-Time %',
            marker_color=colors,
            text=period_stats['on_time_pct'].apply(lambda x: f'{x:.1f}%'),
            textposition='outside',
            textfont=dict(color=THEME['text_color']),
            showlegend=False,
            hovertemplate='%{y:.1f}%<extra></extra>',
        ),
        row=1, col=2
    )
    
    # Summary stats
    total = period_stats['total'].sum()
    worst_period = period_stats.loc[period_stats['on_time_pct'].idxmin(), 'period']
    
    fig.update_layout(
        title=dict(
            text=f'🕐 Commute Period Analysis<br><span style="font-size:14px;color:{THEME["text_color"]}">Last {DAYS_TO_ANALYZE} days | {int(total):,} arrivals | Most delays: {worst_period}</span>',
            font=dict(size=22, color=THEME['text_color']),
            x=0.5,
            xanchor='center'
        ),
        barmode='stack',
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=-0.2,
            xanchor='center',
            x=0.25,
            font=dict(color=THEME['text_color']),
        ),
        plot_bgcolor=THEME['bg_color'],
        paper_bgcolor=THEME['bg_color'],
        height=FIGURE_HEIGHT,
        width=FIGURE_WIDTH,
        margin=dict(l=60, r=40, t=100, b=100),
    )
    
    # Update axes
    for i in [1, 2]:
        fig.update_xaxes(tickfont=dict(color=THEME['text_color']), tickangle=20, row=1, col=i)
    fig.update_yaxes(title='Count', tickfont=dict(color=THEME['text_color']), gridcolor=THEME['grid_color'], row=1, col=1)
    fig.update_yaxes(title='On-Time %', ticksuffix='%', tickfont=dict(color=THEME['text_color']), gridcolor=THEME['grid_color'], range=[0, 100], row=1, col=2)
    
    # Update subplot titles
    for ann in fig.layout.annotations:
        ann.font.color = THEME['text_color']
    
    return fig

fig = create_commute_chart(period_stats, arrivals)
fig.show()

## Weekend vs Weekday Analysis

In [6]:
def analyze_weekend_weekday(arrivals):
    print("=" * 50)
    print("📊 WEEKEND VS WEEKDAY ANALYSIS")
    print("=" * 50)
    
    weekday = arrivals[~arrivals['is_weekend']]
    weekend = arrivals[arrivals['is_weekend']]
    
    wd_pct = (weekday['status'] == 'On Time').mean() * 100
    we_pct = (weekend['status'] == 'On Time').mean() * 100 if len(weekend) > 0 else 0
    
    print(f"\n📅 Weekday: {wd_pct:.1f}% on-time ({len(weekday):,} arrivals)")
    print(f"📅 Weekend: {we_pct:.1f}% on-time ({len(weekend):,} arrivals)")
    
    if we_pct > wd_pct:
        print(f"\n✅ Weekends are {we_pct - wd_pct:.1f}% better than weekdays")
    else:
        print(f"\n⚠️ Weekends are {wd_pct - we_pct:.1f}% worse than weekdays")

analyze_weekend_weekday(arrivals)

📊 WEEKEND VS WEEKDAY ANALYSIS

📅 Weekday: 92.7% on-time (37,647 arrivals)
📅 Weekend: 91.7% on-time (10,871 arrivals)

⚠️ Weekends are 1.0% worse than weekdays


## Export

In [7]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, 'commute_delay.html')
fig.write_html(output_path, include_plotlyjs='cdn')
print(f"✅ Exported to: {output_path}")

✅ Exported to: d:\caltrain-tracker\static\plots\commute_delay.html
